# 03 — Model Training and Evaluation

## 1. Introduction and Objectives

### Objective
The objective of this notebook is to train, evaluate, optimize, and compare six supervised Machine Learning classifiers for predicting recorded firewall actions from network flow features.

### Classification Task
- **Type:** Supervised multiclass classification
- **Target variable:** `Action`
- **Classes (4):** `allow`, `deny`, `drop`, `reset-both`
- **Dataset split (from Notebook 02):** 57,170 distinct flows partitioned into **45,736 training rows** (`allow` = 29,951; `drop` = 9,308; `deny` = 6,434; `reset-both` = 43) and **11,434 held-out test rows** (`allow` = 7,488; `drop` = 2,327; `deny` = 1,608; `reset-both` = 11).

### Candidate Models (Professor Specification)
1. **Dummy Classifier (Zero-R)** — Majority-class baseline
2. **K-Nearest Neighbors (KNN)** — Distance-based classifier
3. **Decision Tree** — Interpretable rule-based tree
4. **Random Forest** — Nonlinear ensemble of trees
5. **Gaussian Naive Bayes** — Probabilistic generative baseline
6. **Support Vector Machine (SVM)** — Margin-based classifier

### Methodological & Leakage Controls
- **No test-set peeking:** The held-out test set (`data/processed/test.csv`) remains untouched until final evaluation in Step 09.
- **In-fold preprocessing:** Scaling and transformations from `src/preprocessing.py` are encapsulated inside a scikit-learn `Pipeline` so they are fitted strictly on each training fold.
- **Group-aware cross-validation:** Identical 11-feature patterns are grouped via `StratifiedGroupKFold(n_splits=5)` to prevent split leakage across training and validation folds.
- **Decision-time leakage ablation:** We compare `full` (11 features), `without_nat` (9 features), and `ports_only` (2 features) scenarios to assess reliance on post-decision counters and NAT attributes.

In [1]:
import numpy as np, pandas as pd, time, json, joblib
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedGroupKFold, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import (classification_report, confusion_matrix, f1_score,balanced_accuracy_score, accuracy_score, roc_auc_score)
RANDOM_STATE = 42